# Week 3: V1 Data Pipeline (ResponsibleAI Guide)
Pathway A: PDF. Main source: S01 NIST AI RMF 1.0.

Flow follows the Topic 3 Lab: Cell 1 → Upload → Cell 3 → Cell 4 → Cell 4A → Cell 5A1/5A2 → B3 → B4.1 → Cell 6A → Cell 6B → Cell 7 → Cell 8A → Cell 9.

## PART A: Source identification and inspection

In [1]:
# ============================================================
# PART A | Cell 1: Set up the workspace (run FIRST)
# Creates the raw and processed folders. Run again after any reset.
# ============================================================
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Short names used by the lab examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)


Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


In [2]:
# ============================================================
# PART A | Cell 2: Copy the PDFs from Google Drive into Colab
# Needed again after every disconnect. Skip the lab's ZIP cell.
# ============================================================
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/AI Engineering/ai_engineering_data")

pdfs = sorted(DRIVE_DIR.glob("*.pdf"))
assert pdfs, "No PDFs found. Check the Drive folder name."

for pdf in pdfs:
    shutil.copy(pdf, RAW_DIR / pdf.name)

print("Copied:", [pdf.name for pdf in pdfs])


Mounted at /content/drive
Copied: ['asean_ai_governance_ethics_2024.pdf', 'malaysia_aige_2024.pdf', 'nist_ai_100-1_ai_rmf_1.0.pdf', 'nist_ai_600-1_genai_profile.pdf', 'unesco_genai_guidance_2023.pdf']


In [3]:
# ============================================================
# PART A | Cell 3 (Upload check): Check the files are really in data/raw
# Shows every file and its size (proof of ingestion).
# ============================================================
from pathlib import Path

RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

all_files = sorted(p for p in RAW.rglob("*") if p.is_file())
print("Input count:", len(all_files))
for p in all_files:
    print(p.relative_to(RAW), "bytes:", p.stat().st_size)

assert all_files, "Upload at least one relevant input file before proceeding."
print("Upload verified.")


Input count: 5
asean_ai_governance_ethics_2024.pdf bytes: 4233765
malaysia_aige_2024.pdf bytes: 9848486
nist_ai_100-1_ai_rmf_1.0.pdf bytes: 1946127
nist_ai_600-1_genai_profile.pdf bytes: 1174643
unesco_genai_guidance_2023.pdf bytes: 1958048
Upload verified.


In [4]:
# ============================================================
# PART A | Step A3, Cell A3: Inspect all 5 PDFs (evidence for the inventory)
# Shows pages, characters, near-empty pages and text samples.
# ============================================================
!pip -q install pypdf
from pypdf import PdfReader

pdf_paths = sorted(RAW_DIR.glob("*.pdf"))
assert pdf_paths, "No PDFs found in data/raw. Run Cell 2 first."

for pdf_path in pdf_paths:
    reader = PdfReader(str(pdf_path))
    if reader.is_encrypted:
        print("File:", pdf_path.name, "is encrypted, skip or check")
        continue

    page_text_list = [(p.extract_text() or "") for p in reader.pages]
    near_empty_pages = [
        i for i, t in enumerate(page_text_list, start=1) if len(t.strip()) < 50
    ]
    first_text = next((t for t in page_text_list if len(t.strip()) > 200), "")
    middle = page_text_list[len(page_text_list) // 2]

    print("=" * 60)
    print("File:", pdf_path.name)
    print("Pages:", len(page_text_list))
    print("Characters extracted:", sum(len(t) for t in page_text_list))
    print("Near-empty pages:", len(near_empty_pages), near_empty_pages)
    print("Sample (first text page):", repr(first_text[:300]))
    print("Sample (middle page):", repr(middle[:300]))


File: asean_ai_governance_ethics_2024.pdf
Pages: 87
Characters extracted: 191311
Near-empty pages: 9 [1, 6, 10, 17, 46, 53, 58, 60, 73]
Sample (first text page): 'Contents\nExecutive Summary\nA. Introduction\n1. Objectives\n2. Assumptions\n3. Target Audience\n4. Definitions\nB. Guiding Principles for the Framework\n1. Transparency and Explainability\n2. Fairness and Equity\n3. Security and Safety\n4. Human-centricity\n5. Privacy and Data Governance\n6. Accountability and '
Sample (middle page): 'GOJEK\nIllustration on stakeholder interaction and communication\nGojek is an Indonesian on-demand multi-service platform and digital payment group based in Jakarta. Al is\nleveraged for user-base growth and maintaining engagement of consumers through automated allocation of\npromotions, under budget co'
File: malaysia_aige_2024.pdf
Pages: 140
Characters extracted: 258339
Near-empty pages: 5 [3, 10, 40, 114, 140]
Sample (first text page): 'Published by: \nMinistry of Science, Technology and Inn

## PART B: Data preparation, comparison and lineage

In [5]:
# ============================================================
# PART B | Cell 4: Helper functions (select source, hash, manifest)
# Required for everyone. Run once after Cell 3.
# ============================================================
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    # Pick one exact file. Never take "the first file" by extension.
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source. Set relative_path to an exact file under RAW_DIR.")
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

print("Cell 4 completed successfully.")
print("Available functions: select_source(), source_hash(), save_manifest()")


Cell 4 completed successfully.
Available functions: select_source(), source_hash(), save_manifest()


In [6]:
# ============================================================
# PART B | Cell 4A: Lineage registry
# Prepares the place where source-to-output links are recorded.
# Run once, right after Cell 4.
# ============================================================
from pathlib import Path

lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()

    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError("Replace the placeholder with the actual method.")

    # Source must be in raw, output must be in processed
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())

    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {"source": source, "output": output, "processing": processing}
    print("Registered:", source.name, "->", output.name)

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))


Common lineage registry ready.
Registered entries: 0


In [7]:
# ============================================================
# PART B | Cell 5A1: Read the main PDF (S01) and make the two lab chunk sets
# Lab pathway A1: fixed-size vs paragraph-aware. Only the file name is changed.
# Output variables used later: f, pages, clean, fixed, structured
# ============================================================
!pip -q install pypdf
from pypdf import PdfReader
import re

# Step 1: Select the exact source file (S01: NIST AI RMF 1.0)
f = select_source(relative_path="nist_ai_100-1_ai_rmf_1.0.pdf")

# Step 2: Extract text. pages = list of (page number, page text)
if f.suffix.lower() == ".pdf":
    reader = PdfReader(str(f))
    pages = [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]
elif f.suffix.lower() == ".txt":
    pages = [(1, f.read_text(encoding="utf-8"))]
else:
    raise ValueError("Pathway A needs a PDF or TXT source")

# Step 3: Basic cleaning (collapse extra spaces and tabs)
text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages/segments:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters:", len(clean))
print("Content sample:", repr(clean[:300]))
assert clean, "No usable text: check extraction or OCR needs."

# Step 4: Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [s[i:i + size] for i in range(0, len(s), size)]

# Step 5: Paragraph-aware chunking (splits on blank lines)
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))
        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts

# Step 6: Compare the two lab approaches
fixed = fixed_chunks(clean, size=500)
structured = paragraph_chunks(clean, size=500)

print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))
print("\nFixed sample:", repr(fixed[0][:180]))
print("\nParagraph-aware sample:", repr(structured[0][:180]))
print("\nPATHWAY A1 completed successfully.")


Source: nist_ai_100-1_ai_rmf_1.0.pdf
Pages/segments: 48
Extracted characters: 106493
Cleaned characters: 106492
Content sample: 'NIST AI 100-1\nArtificial Intelligence Risk Management\nFramework (AI RMF 1.0)\n\nNIST AI 100-1\nArtificial Intelligence Risk Management\nFramework (AI RMF 1.0)\nThis publication is available free of charge from:\nhttps://doi.org/10.6028/NIST.AI.100-1\nJanuary 2023\nU.S. Department of Commerce\nGina M. Raimond'

--- Chunking Comparison ---
Fixed chunks: 213
Paragraph-aware chunks: 214

Fixed sample: 'NIST AI 100-1\nArtificial Intelligence Risk Management\nFramework (AI RMF 1.0)\n\nNIST AI 100-1\nArtificial Intelligence Risk Management\nFramework (AI RMF 1.0)\nThis publication is avail'

Paragraph-aware sample: 'NIST AI 100-1\nArtificial Intelligence Risk Management\nFramework (AI RMF 1.0)'

PATHWAY A1 completed successfully.


In [8]:
# ============================================================
# PART B | Cell 5A1b: Page-aware chunking (my adaptation) + comparison
# Keeps the PDF page number, removes the repeated page header,
# and never cuts a word in the middle. Needs Cell 5A1 first.
# Output variables used later: page_records, approach
# ============================================================
from collections import Counter

# Safety check: pages must come from Cell 5A1 (a list of (page, text) pairs)
assert isinstance(pages[0], tuple), "Run Cell 5A1 again first (pages was overwritten)."

# Step 1: Find header lines repeated at the top of many pages
first_lines = Counter(
    page_text.strip().split("\n")[0].strip()
    for _, page_text in pages if page_text.strip()
)
REPEATED_HEADERS = {line for line, n in first_lines.items() if n >= 10}
print("Repeated headers to remove:", REPEATED_HEADERS)

# Step 2: Page-aware chunking
# - a chunk never crosses a page
# - a line ending in "-" is joined to the next line without a space
#   (the hyphen is kept, because some hyphens are real, like "red-teaming")
# - a chunk may go a few characters over 500 to finish a hyphenated word
def page_aware_chunks(page_list, size=500):
    out = []
    forced_cuts = 0
    for page_num, page_text in page_list:
        lines = [re.sub(r"[ \t]+", " ", ln).strip() for ln in page_text.split("\n")]
        lines = [ln for ln in lines if ln and ln not in REPEATED_HEADERS]
        current = ""
        for ln in lines:
            if len(ln) > size:
                forced_cuts += 1
                pieces = [ln[i:i + size] for i in range(0, len(ln), size)]
            else:
                pieces = [ln]
            for piece in pieces:
                joiner = "" if current.endswith("-") else " "
                # Start a new chunk only if the chunk is full AND does not end
                # with "-" (so a hyphenated word is not split across chunks)
                if (current and not current.endswith("-")
                        and len(current) + len(joiner) + len(piece) > size):
                    out.append({"page": page_num, "text": current})
                    current = piece
                else:
                    current = current + joiner + piece if current else piece
        if current:
            out.append({"page": page_num, "text": current})
    return out, forced_cuts

page_records, forced_cuts = page_aware_chunks(pages, size=500)
page_chunk_texts = [r["text"] for r in page_records]
approach = "page-aware-line-packed-char-500"

# Step 3: Compare the three approaches with real numbers
def summary(name, chunks, has_page):
    lens = [len(c) for c in chunks]
    header_hits = sum(c.count(h) for c in chunks for h in REPEATED_HEADERS)
    print(f"{name}:")
    print("  chunks:", len(chunks))
    print("  chunks with a page number:", len(chunks) if has_page else 0)
    print("  average / shortest length:", round(sum(lens) / len(lens)), "/", min(lens))
    print("  repeated header inside chunks (approx):", header_hits)

print("\n--- Comparison (same document, same size 500) ---")
summary("Fixed (lab)", fixed, False)
summary("Paragraph-aware (lab)", structured, False)
summary("Page-aware (adapted)", page_chunk_texts, True)

print("\nParagraph-aware identical to fixed:", structured == fixed)
mid_word = sum(
    1 for a, b in zip(fixed, fixed[1:])
    if a and b and a[-1].isalnum() and b[0].isalnum()
)
print("Fixed: boundaries that cut a word in the middle:", mid_word)
print("Page-aware: lines longer than the chunk size (forced cuts):", forced_cuts)
print("\nPage-aware sample:", page_records[min(10, len(page_records) - 1)])


Repeated headers to remove: {'NIST AI 100-1 AI RMF 1.0'}

--- Comparison (same document, same size 500) ---
Fixed (lab):
  chunks: 213
  chunks with a page number: 0
  average / shortest length: 500 / 492
  repeated header inside chunks (approx): 43
Paragraph-aware (lab):
  chunks: 214
  chunks with a page number: 0
  average / shortest length: 498 / 76
  repeated header inside chunks (approx): 42
Page-aware (adapted):
  chunks: 238
  chunks with a page number: 238
  average / shortest length: 441 / 6
  repeated header inside chunks (approx): 0

Paragraph-aware identical to fixed: False
Fixed: boundaries that cut a word in the middle: 152
Page-aware: lines longer than the chunk size (forced cuts): 0

Page-aware sample: {'page': 5, 'text': 'context and are performed throughout the AI system lifecycle. See Figure 3 for representative AI actors. 10 Fig. 3 AI actors across AI lifecycle stages. See Appendix A for detailed descrip-tions of AI actor tasks, including details about testing, eva

In [9]:
# ============================================================
# PART B | Cell 5A2: Export the chosen chunks (adapted: keeps page numbers)
# Writes data/processed/prepared_text.jsonl. Needs Cell 5A1b first.
# Output variables used later: prepared_path, approach
# ============================================================
import json

file_sha = source_hash(f)  # calculate the file fingerprint once

records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": item["text"],
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": file_sha,
        "processing": approach,
        "page": item["page"]  # PDF page number (1 = first page of the file)
    }
    for i, item in enumerate(page_records, 1)
]

prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

# Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert all(r["text"].strip() for r in records), "One or more chunks are empty"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [json.loads(line) for line in file if line.strip()]
assert len(exported_records) == len(records), "Exported record count does not match"
assert all(r["page"] is not None for r in exported_records), "A record has no page"

print("PATHWAY A2 (adapted) completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])


PATHWAY A2 (adapted) completed successfully.
Selected approach: page-aware-line-packed-char-500
Total chunks exported: 238
Prepared file: /content/week3_project/data/processed/prepared_text.jsonl
File size: 164124 bytes
Validation: PASS
First record: {'record_id': 'nist_ai_100-1_ai_rmf_1.0-0001', 'text': 'NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0)', 'source': 'nist_ai_100-1_ai_rmf_1.0.pdf', 'source_sha256': '7576edb531d9848825814ee88e28b1795d3a84b435b4b797d3670eafdc4a89f1', 'processing': 'page-aware-line-packed-char-500', 'page': 1}


In [10]:
# ============================================================
# PART B | Cell B3: Basic output check (required)
# Confirms the prepared file exists, is not empty, and looks right.
# ============================================================
assert prepared_path.exists(), "Prepared output missing."
assert prepared_path.stat().st_size > 0, "Prepared output is empty."
print("Output:", prepared_path, "Size:", prepared_path.stat().st_size)
print("Validation: file exists and is non-empty; verify content manually too.")

# Look at real records (a non-empty file alone is not proof)
print("\nRecord 3:", exported_records[2])
print("Last record:", exported_records[-1])
print("Pages covered:", min(r["page"] for r in exported_records), "to", max(r["page"] for r in exported_records))


Output: /content/week3_project/data/processed/prepared_text.jsonl Size: 164124
Validation: file exists and is non-empty; verify content manually too.

Record 3: {'record_id': 'nist_ai_100-1_ai_rmf_1.0-0003', 'text': 'Certain commercial entities, equipment, or materials may be identified in this document in order to describe an experimental procedure or concept adequately. Such identification is not intended to imply recommenda-tion or endorsement by the National Institute of Standards and Technology, nor is it intended to imply that the entities, materials, or equipment are necessarily the best available for the purpose. This publication is available free of charge from: https://doi.org/10.6028/NIST.AI.100-1', 'source': 'nist_ai_100-1_ai_rmf_1.0.pdf', 'source_sha256': '7576edb531d9848825814ee88e28b1795d3a84b435b4b797d3670eafdc4a89f1', 'processing': 'page-aware-line-packed-char-500', 'page': 3}
Last record: {'record_id': 'nist_ai_100-1_ai_rmf_1.0-0238', 'text': 'This publication is avai

In [11]:
# ============================================================
# PART B | Cell B4.1: Register the lineage (source -> prepared output)
# Links the PDF to prepared_text.jsonl and records how it was made.
# ============================================================
register_lineage(
    source=f,
    output=prepared_path,
    processing=(
        approach
        + "; removed repeated page header; joined wrapped lines (hyphens kept); "
        + "page = PDF page number"
    )
)
print("Registered entries:", len(lineage_entries))


Registered: nist_ai_100-1_ai_rmf_1.0.pdf -> prepared_text.jsonl
Registered entries: 1


In [12]:
# ============================================================
# PART B | Cell 6A: Create the common lineage manifest
# Writes data/week3_manifest.json for everything registered.
# ============================================================
assert lineage_entries, "No lineage entries registered. Register at least one pathway first."

lineage_records = []
for i, entry in enumerate(lineage_entries.values(), 1):
    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })

manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print("Saved to:", BASE_DIR / "data" / "week3_manifest.json")


Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 1
Saved to: /content/week3_project/data/week3_manifest.json


In [13]:
# ============================================================
# PART B | Cell 6B: Verify the manifest
# Checks the files exist and the SHA-256 fingerprints match.
# ============================================================
import json

manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), "Common lineage manifest not found."

with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)

saved_records = saved_manifest.get("records", [])
assert saved_records, "No lineage records found in saved manifest."
assert len(saved_records) == len(lineage_entries), "Saved lineage count does not match registration."

for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], f"Source checksum mismatch: {source}"
    assert source_hash(output) == record["prepared_sha256"], f"Prepared output checksum mismatch: {output}"

print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)


COMMON LINEAGE VALIDATION: PASS
Verified records: 1
Manifest: /content/week3_project/data/week3_manifest.json


## PART C: Build and document the V1 data pipeline
Cell 7 (common utilities) → Cell 8A (PDF pathway) → Cell 9 (repeatability check).

In [14]:
# ============================================================
# PART C | Cell 7: Common V1 pipeline utilities (REQUIRED, run once)
# From the lab. Defines the helper functions used by Cell 8A and Cell 9.
# No data is processed in this cell.
# ============================================================
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Standard project paths (same as Cell 1)
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}


# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path

    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )

    return source


# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )
    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )

    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )

        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified — check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"

    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)

    return record


# Step 5: Register and run a selected pathway
def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()

    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }
    return first


print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")


Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


In [15]:
# ============================================================
# PART C | Cell 8A: PDF pipeline for S01 (CONDITIONAL REQUIRED)
# Runs ingest -> inspect -> prepare -> validate -> record lineage.
# Uses the SAME choices as Part B: page-aware chunks, size 500,
# repeated header removed, hyphens kept, PDF page number kept.
# Adapted from the lab: the lab splitter writes "page": None,
# but my project needs page numbers for citations (FR3 / RAI1).
# Needs Cell 7 first.
# ============================================================
!pip -q install pypdf

import re
import json
from collections import Counter
from pypdf import PdfReader

# Step 1: Select the source and the Part B configuration
SOURCE_A = resolve_source("nist_ai_100-1_ai_rmf_1.0.pdf")

CONFIG_A = {
    "chunk_size": 500,
    "method": "page-aware",
    "header_min_pages": 10   # a first line repeated on 10+ pages = page header
}


# Step 2: Helper functions (own names, so Part B cells are not changed)
def read_pages(src):
    """Return a list of (PDF page number, page text)."""
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        return [(i + 1, p.extract_text() or "") for i, p in enumerate(reader.pages)]
    if src.suffix.lower() == ".txt":
        return [(1, src.read_text(encoding="utf-8"))]
    raise ValueError("Expected a PDF or TXT source")


def find_repeated_headers(pages, min_pages):
    first_lines = Counter(
        text.strip().split("\n")[0].strip()
        for _, text in pages if text.strip()
    )
    return {line for line, n in first_lines.items() if n >= min_pages}


def v1_page_chunks(pages, size, headers):
    out = []
    for page_num, page_text in pages:
        lines = [re.sub(r"[ \t]+", " ", ln).strip() for ln in page_text.split("\n")]
        lines = [ln for ln in lines if ln and ln not in headers]
        current = ""
        for ln in lines:
            if len(ln) > size:
                pieces = [ln[i:i + size] for i in range(0, len(ln), size)]
            else:
                pieces = [ln]
            for piece in pieces:
                joiner = "" if current.endswith("-") else " "
                if (current and not current.endswith("-")
                        and len(current) + len(joiner) + len(piece) > size):
                    out.append({"page": page_num, "text": current})
                    current = piece
                else:
                    current = current + joiner + piece if current else piece
        if current:
            out.append({"page": page_num, "text": current})
    return out


# Step 3: Inspect the document
def inspect_documents(inputs, cfg):
    src = inputs[0]
    pages = read_pages(src)
    text = "\n".join(t for _, t in pages)

    if not text.strip():
        raise ValueError("No extractable text found. Scanned PDFs may require OCR.")

    return {
        "source": src.name,
        "characters": len(text),
        "pages": len(pages),
        "near_empty_pages": [n for n, t in pages if len(t.strip()) < 50],
        "repeated_headers": sorted(find_repeated_headers(pages, cfg["header_min_pages"]))
    }


# Step 4: Prepare the document (chunks + page numbers)
def prepare_documents(inputs, out, cfg):
    src = inputs[0]
    size = int(cfg["chunk_size"])
    method = cfg["method"]

    if size <= 0:
        raise ValueError("Chunk size must be positive")
    if method != "page-aware":
        raise ValueError("This project uses 'page-aware' (same as Part B).")

    pages = read_pages(src)
    headers = find_repeated_headers(pages, cfg["header_min_pages"])
    chunks = v1_page_chunks(pages, size, headers)

    if not chunks:
        raise ValueError("No text chunks produced")

    method_note = (
        f"{method} text chunks of {size} characters; removed repeated page header; "
        "joined wrapped lines (hyphens kept); page = PDF page number"
    )
    file_sha = sha256_file(src)

    # Step 5: Store prepared document records
    target = out / "prepared_text.jsonl"
    with target.open("w", encoding="utf-8") as handle:
        for i, item in enumerate(chunks, 1):
            record = {
                "record_id": f"{src.stem}-{i:04d}",
                "text": item["text"],
                "source": str(src.relative_to(RAW_DIR)),
                "source_sha256": file_sha,
                "processing": method_note,
                "page": item["page"]
            }
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    # Check the file we just wrote (every record needs text and a page)
    with target.open("r", encoding="utf-8") as handle:
        written = [json.loads(line) for line in handle if line.strip()]
    assert len(written) == len(chunks), "Written record count does not match"
    assert all(r["text"].strip() for r in written), "A chunk is empty"
    assert all(r["page"] is not None for r in written), "A record has no page"

    # Step 6: Return the source-to-output mapping
    mapping = [{"source": str(src), "output": str(target)}]
    return [target], mapping, method_note


# Step 7: Execute and register the pipeline
run_A = register_and_run(
    "documents",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)

print("Cell 8A completed successfully.")
print("Inspection:", run_A["inspection"])


SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
Cell 8A completed successfully.
Inspection: {'source': 'nist_ai_100-1_ai_rmf_1.0.pdf', 'characters': 106493, 'pages': 48, 'near_empty_pages': [], 'repeated_headers': ['NIST AI 100-1 AI RMF 1.0']}


In [16]:
# ============================================================
# PART C | Cell 9: Repeatability and provenance checks (REQUIRED)
# Runs each registered pipeline a second time and compares the
# source fingerprints, output fingerprints and mappings.
# Run after Cell 8A.
# ============================================================

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)
print("--- Repeatability Verification ---")

# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]

    # Execute the same pipeline again
    second = entry["runner"]()

    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )

    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )

    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )

    print(
        f"PASS: {kind} — source fingerprints, "
        "prepared outputs and mappings match"
    )

# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")


--- Repeatability Verification ---
SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


In [17]:
# ============================================================
# PART C | Step C1 evidence (extra check, not a lab cell)
# Looks at real records and compares Part C with the Part B output.
# The lab asks us to explain any difference from Part B.
# ============================================================
v1_path = PIPE_DIR / "documents" / "prepared_text.jsonl"
with v1_path.open("r", encoding="utf-8") as file:
    v1_records = [json.loads(line) for line in file if line.strip()]

print("Part C prepared file:", v1_path)
print("Part C records:", len(v1_records), "| Part B records:", len(exported_records))
print("Same text in the same order:",
      [r["text"] for r in v1_records] == [r["text"] for r in exported_records])
print("Same page numbers:",
      [r["page"] for r in v1_records] == [r["page"] for r in exported_records])
print("Pages covered:", min(r["page"] for r in v1_records), "to", max(r["page"] for r in v1_records))
print("\nRepresentative record:", v1_records[len(v1_records) // 2])
print("Pipeline manifest:", PIPE_DIR / "documents" / "pipeline_manifest.json")


Part C prepared file: /content/week3_project/data/pipeline_v1/documents/prepared_text.jsonl
Part C records: 238 | Part B records: 238
Same text in the same order: True
Same page numbers: True
Pages covered: 1 to 48

Representative record: {'record_id': 'nist_ai_100-1_ai_rmf_1.0-0120', 'text': 'interests. Playbook users can create tailored guidance selected from suggested material for their own use and contribute their suggestions for sharing with the broader community. Along with the AI RMF, the Playbook is part of the NIST Trustworthy and Responsible AI Resource Center. Framework users may apply these functions as best suits their needs for managing AI risks based on their resources and capabilities. Some organizations may choose', 'source': 'nist_ai_100-1_ai_rmf_1.0.pdf', 'source_sha256': '7576edb531d9848825814ee88e28b1795d3a84b435b4b797d3670eafdc4a89f1', 'processing': 'page-aware text chunks of 500 characters; removed repeated page header; joined wrapped lines (hyphens kept); page =

In [18]:
# ============================================================
# PART C | Step C4a: Zip all Week 3 outputs and download
# Run after Cell 9. Raw PDFs are NOT included (S03 is copyrighted).
# ============================================================
import zipfile
from pathlib import Path
from google.colab import files

BASE_DIR = Path("/content/week3_project")
ZIP_PATH = Path("/content/week3_outputs.zip")

items = [
    BASE_DIR / "data" / "week3_manifest.json",   # Part B manifest
    BASE_DIR / "data" / "processed",             # Part B output
    BASE_DIR / "data" / "pipeline_v1",           # Part C output + manifest
]

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for item in items:
        assert item.exists(), f"Missing: {item}"
        for p in ([item] if item.is_file() else sorted(item.rglob("*"))):
            if p.is_file():
                z.write(p, p.relative_to(BASE_DIR))
                print("Added:", p.relative_to(BASE_DIR))

print("Zip size:", ZIP_PATH.stat().st_size, "bytes")

# Download to your computer
files.download(str(ZIP_PATH))

Added: data/week3_manifest.json
Added: data/processed/prepared_text.jsonl
Added: data/pipeline_v1/documents/pipeline_manifest.json
Added: data/pipeline_v1/documents/prepared_text.jsonl
Zip size: 73946 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>